# Оптимизация с ограничением на пик
Минимизация смертности при условии peak <= 0.3.

In [1]:
using DrWatson
@quickactivate "project"

using BlackBoxOptim
using CSV
using DataFrames

include(srcdir("sir_model.jl"))

script_name = splitext(basename(PROGRAM_FILE))[1]
mkpath(datadir(script_name))

const OPTIMIZATION_SEEDS = (101, 202, 303)

function constrained_objective(x)
    beta_und = x[1]
    detection_time = round(Int, x[2])
    death_rate = x[3]
    mean_peak = 0.0
    mean_death = 0.0
    for seed in OPTIMIZATION_SEEDS
        model = initialize_sir(
            Ns = [1000, 1000, 1000],
            Is = [1, 0, 0],
            beta_und = beta_und,
            beta_det = beta_und / 10,
            infection_period = 14,
            detection_time = clamp(detection_time, 2, 12),
            death_rate = clamp(death_rate, 0.0, 0.1),
            reinfection_probability = 0.1,
            migration_intensity = 0.2,
            seed = seed,
        )
        df = simulate_sir!(model, 60)
        metrics = summarize_dynamics(df, 3000)
        mean_peak += metrics.peak
        mean_death += metrics.death_fraction
    end
    mean_peak /= length(OPTIMIZATION_SEEDS)
    mean_death /= length(OPTIMIZATION_SEEDS)
    penalty = mean_peak > 0.3 ? 100 * (mean_peak - 0.3)^2 : 0.0
    return mean_death + penalty
end

res = bboptimize(constrained_objective;
    SearchRange = [(0.35, 0.8), (2.0, 12.0), (0.02, 0.08)],
    NumDimensions = 3,
    MaxSteps = 25,
    TraceMode = :verbose,
)

best = best_candidate(res)
beta_und = best[1]
detection_time = round(Int, best[2])
death_rate = best[3]
mean_peak, mean_death = let peak_acc = 0.0, death_acc = 0.0
    for seed in OPTIMIZATION_SEEDS
        model = initialize_sir(
            Ns = [1000, 1000, 1000],
            Is = [1, 0, 0],
            beta_und = beta_und,
            beta_det = beta_und / 10,
            infection_period = 14,
            detection_time = clamp(detection_time, 2, 12),
            death_rate = clamp(death_rate, 0.0, 0.1),
            reinfection_probability = 0.1,
            migration_intensity = 0.2,
            seed = seed,
        )
        df = simulate_sir!(model, 60)
        metrics = summarize_dynamics(df, 3000)
        peak_acc += metrics.peak
        death_acc += metrics.death_fraction
    end
    (peak_acc / length(OPTIMIZATION_SEEDS), death_acc / length(OPTIMIZATION_SEEDS))
end

out = DataFrame(
    beta_und = [beta_und],
    detection_time = [clamp(detection_time, 2, 12)],
    death_rate = [clamp(death_rate, 0.0, 0.1)],
    objective = [best_fitness(res)],
    peak_infected = [mean_peak],
    death_fraction = [mean_death],
    satisfies_constraint = [mean_peak <= 0.3],
)
CSV.write(datadir(script_name, "optimization_with_constraint.csv"), out)
println(out)
println("saved: ", datadir(script_name, "optimization_with_constraint.csv"))

Starting optimization with optimizer DiffEvoOpt{FitPopulation{Float64}, RadiusLimitedSelector, BlackBoxOptim.AdaptiveDiffEvoRandBin{3}, RandomBound{ContinuousRectSearchSpace}}
0.00 secs, 0 evals, 0 steps


DE modify state:
0.58 secs, 2 evals, 1 steps

, fitness=20.031967901
DE modify state:
1.13 secs, 4 evals, 2 steps

, fitness=16.813504938
DE modify state:
1.64 secs, 6 evals, 3 steps

, improv/step: 0.333 (last = 1.0000), fitness=16.813504938
DE modify state:
2.75 secs, 10 evals, 5 steps, improv/step: 0.200 (last = 0.0000)

...

, improv/step: 0.522 (last = 0.0000), fitness=0.000444444
DE modify state:
12.94 secs, 48 evals, 25 steps, improv/step: 0.520 (last = 0.5000)

, fitness=0.000444444
DE modify state:

Optimization stopped after 26 steps and 13.48 seconds


Termination reason: Max number of steps (25) reached
Steps per second = 1.93
Function evals per second = 3.71
Improvements/step = 0.52000
Total function evaluations = 50


Best candidate found: 

[0.410295, 2.61016, 0.0246053]

Fitness: 0.000444444

1×7 DataFrame
 Row │ beta_und  detection_time  death_rate  objective    peak_infected  death_fraction  satisfies_constraint 
     │ Float64   Int64           Float64     Float64      Float64        Float64         Bool                 
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ 0.410295               3   0.0246053  0.000444444     0.00922222     0.000444444                  true


saved: /home/lilidji/github-push-work/labs/lab04/project/data/optimization_with_constraint.csv
